# Geometry-MMALS G1 v1.0.7
## Context-Bottleneck Mediation Experiment

**Scientific status:** controlled protocol implementation and development experiment.  
**Accepted claim:** C0 pipeline/protocol integrity only after successful execution.  
**Not claimed:** C1-C6 qualification, successful context mediation, adaptive-route superiority, backward transfer, replay-free continual learning, domain-general geometry, or quantum advantage.

v1.0.7 addresses the strongest unresolved result from v1.0.6: the standard router was dominated by the direct sensory representation `z0`, while context zeroing or shuffling had almost no predictive effect. The central question is therefore:

> Does context mediation fail because the router takes an architectural shortcut through `z0`, or because the inferred context itself lacks useful geometric information?

The primary intervention is a dedicated router that **never receives `z0`**. A capacity-matched sensory-only router is used as the decisive control.

## Research history

| Version | Main question | Outcome |
|---|---|---|
| v1.0.0 | Can internal MMALS geometry be specified and falsified? | Evidence ladder and C0 scaffold. |
| v1.0.1 | Can claims, gates, notation and memory contracts be audited? | C0-only discipline and pilot gates. |
| v1.0.2 | Can route geometry train without numerical corruption? | Stable Fisher-Rao/chordal implementation. |
| v1.0.3 | Does the optimization unit contain genuine cross-angle evidence? | Same-source multi-angle protocol. |
| v1.0.4 | Does geometry add value under matched initialization and compute? | Small route-order effect; anchor more useful. |
| v1.0.5 | Is context mediating the route, and how strong is curriculum order? | Direct `z0` shortcut exposed. |
| v1.0.6 | Does adaptive routing beat static routes under same-final-task curricula? | Route geometry measurable; context bypass and curriculum sensitivity remain. |
| **v1.0.7** | **Can a forced context bottleneck recover context-mediated geometry?** | Implemented here; no result claimed before execution. |

## Tracked changes in v1.0.7

- **CHG-107-01** - Added `ContextBottleneckRouter`, which accepts inferred context only.
- **CHG-107-02** - Added a capacity-matched `SensoryBottleneckRouter` control.
- **CHG-107-03** - Removed the proposed stop-gradient control because the encoder is already frozen and detaching `z0` does not prevent router weights from using `z0` values.
- **CHG-107-04** - Reduced the core experiment to six targeted, equal-compute variants.
- **CHG-107-05** - Added explicit `bottleneck_geo_effect`, `bottleneck_vs_standard`, `sensory_geo_effect`, and `context_vs_sensory_geo` contrasts.
- **CHG-107-06** - Extended mediation exports with context/z0 route shifts, accuracy drops, dependency ratio, and `context_is_primary`.
- **CHG-107-07** - Added capacity and parameter-count audit for route policies.
- **CHG-107-08** - Retained C1 trained, C2 interpolation, and extrapolation partitions separately.
- **CHG-107-09** - Restricted counterbalanced curriculum reruns to the primary bottleneck+geometry method.
- **CHG-107-10** - Added reviewer-report and LaTeX-source references to the immutable run manifest.
- **CHG-107-11** - Added package/notebook/version/source-hash integrity checks.
- **CHG-107-12** - Preserved strict non-promotion of C1-C6 from a development run.

## 0. Setup and release-integrity boundary

In [ ]:
import os
import sys
import shutil
import importlib
import subprocess
from pathlib import Path

EXPECTED_PACKAGE_VERSION = "1.0.7"
REPO_URL = "https://github.com/gharbonnier78/geometry-mmalls-g1.git"
REPO_DIR = Path("/content/geometry-mmalls-g1")
SRC_DIR = REPO_DIR / "src"
FORCE_REFRESH = False

if FORCE_REFRESH and REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

assert (REPO_DIR / "pyproject.toml").exists(), f"Missing pyproject.toml in {REPO_DIR}"
assert (SRC_DIR / "geometry_mmalls" / "__init__.py").exists()
os.chdir(REPO_DIR)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-cache-dir", "--no-deps", "-e", str(REPO_DIR)],
    check=True,
)

src_path = str(SRC_DIR)
if src_path not in sys.path:
    sys.path.insert(0, src_path)
for module_name in list(sys.modules):
    if module_name == "geometry_mmalls" or module_name.startswith("geometry_mmalls."):
        del sys.modules[module_name]
importlib.invalidate_caches()
import geometry_mmalls

loaded_from = Path(geometry_mmalls.__file__).resolve()
assert loaded_from.parent == (SRC_DIR / "geometry_mmalls").resolve()
if geometry_mmalls.__version__ != EXPECTED_PACKAGE_VERSION:
    raise RuntimeError(
        f"Loaded geometry_mmalls {geometry_mmalls.__version__}; v{EXPECTED_PACKAGE_VERSION} is required. "
        "Push/pull the v1.0.7 package and rerun with FORCE_REFRESH=True."
    )
print("Python:", sys.executable)
print("Repository:", REPO_DIR)
print("Package:", loaded_from)
print("Version:", geometry_mmalls.__version__)

In [ ]:
from pathlib import Path
from types import SimpleNamespace
import copy
import hashlib
import json
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import yaml
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import MNIST

from geometry_mmalls.data import FixedAngleMNIST, MultiAngleMNIST
from geometry_mmalls.geometry import fisher_rao_distance, paired_route_geometry_loss
from geometry_mmalls.metrics import bootstrap_mean_ci, centroid_geometry_scores, grouped_geometry_scores
from geometry_mmalls.model import GeometryMMALS, SmallConvEncoder

NOTEBOOK_VERSION = "1.0.7"
assert geometry_mmalls.__version__ == NOTEBOOK_VERSION
base_config = yaml.safe_load(Path("configs/rotated_mnist_g1.yaml").read_text())
protocol_config = yaml.safe_load(Path("configs/rotated_mnist_g1_v107.yaml").read_text())
config = base_config

RUN_MODE = "development"  # smoke, development, qualification
SEED = 0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RUN_COUNTERBALANCED_CURRICULA = RUN_MODE != "smoke"

train_angles = list(map(float, config["data"]["train_angles"]))
interp_angles = list(map(float, config["data"]["interpolation_angles"]))
extra_angles = list(map(float, config["data"]["extrapolation_angles"]))
all_eval_angles = train_angles + interp_angles + extra_angles
COUNTERBALANCED_CURRICULA = {
    k: list(map(float, v)) for k, v in protocol_config["counterbalanced_curricula"].items()
}
PRIMARY_CURRICULUM_NAME = protocol_config["primary_curriculum_name"]
PRIMARY_CURRICULUM = COUNTERBALANCED_CURRICULA[PRIMARY_CURRICULUM_NAME]

geo_weight = float(config["losses"]["route_geometry"])
METHOD_SPECS = {
    "standard_anchor_no_geo": {
        "paired": True, "geometry_weight": 0.0, "anchor_weight": 0.10,
        "router_mode": "standard",
    },
    "uniform_static_anchor": {
        "paired": True, "geometry_weight": 0.0, "anchor_weight": 0.10,
        "router_mode": "uniform_static",
    },
    "context_bottleneck_no_geo": {
        "paired": True, "geometry_weight": 0.0, "anchor_weight": 0.10,
        "router_mode": "context_bottleneck",
    },
    "context_bottleneck_geo": {
        "paired": True, "geometry_weight": geo_weight, "anchor_weight": 0.10,
        "router_mode": "context_bottleneck",
    },
    "sensory_bottleneck_no_geo": {
        "paired": True, "geometry_weight": 0.0, "anchor_weight": 0.10,
        "router_mode": "sensory_bottleneck",
    },
    "sensory_bottleneck_geo": {
        "paired": True, "geometry_weight": geo_weight, "anchor_weight": 0.10,
        "router_mode": "sensory_bottleneck",
    },
}
METHODS = list(METHOD_SPECS)

if RUN_MODE == "smoke":
    TRAIN_SOURCE_LIMIT, TEST_SOURCE_LIMIT = 64, 64
    SENSORY_SOURCE_LIMIT, SENSORY_TEST_LIMIT = 256, 128
    SENSORY_EPOCHS, STAGE_EPOCHS, BOOTSTRAP_SAMPLES = 1, 1, 100
    SOURCE_BATCH_SIZE, SENSORY_GATE = 32, 0.0
elif RUN_MODE == "development":
    TRAIN_SOURCE_LIMIT, TEST_SOURCE_LIMIT = 512, 256
    SENSORY_SOURCE_LIMIT, SENSORY_TEST_LIMIT = 2000, 1000
    SENSORY_EPOCHS, STAGE_EPOCHS, BOOTSTRAP_SAMPLES = 2, 2, 500
    SOURCE_BATCH_SIZE, SENSORY_GATE = 64, 0.85
elif RUN_MODE == "qualification":
    TRAIN_SOURCE_LIMIT = int(config["paired_protocol"]["full_source_limit"])
    TEST_SOURCE_LIMIT = 2000
    SENSORY_SOURCE_LIMIT, SENSORY_TEST_LIMIT = 12000, 5000
    SENSORY_EPOCHS = int(config["training"]["sensory_pretrain_epochs"])
    STAGE_EPOCHS = int(config["training"]["stage_epochs"])
    BOOTSTRAP_SAMPLES = int(config["metrics"]["bootstrap_samples"])
    SOURCE_BATCH_SIZE = int(config["paired_protocol"]["source_batch_size"])
    SENSORY_GATE = 0.95
else:
    raise ValueError("RUN_MODE must be smoke, development, or qualification")

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print({
    "notebook_version": NOTEBOOK_VERSION,
    "run_mode": RUN_MODE,
    "device": str(DEVICE),
    "methods": METHODS,
    "primary_curriculum": PRIMARY_CURRICULUM,
})

## 1. Numerical, structural, and capacity self-tests

In [ ]:
# Stable paired geometry backward gate.
_probe_logits = torch.randn(8, 5, 4, requires_grad=True)
_probe_routes = torch.softmax(_probe_logits, dim=-1)
_probe_angles = torch.tensor([-60.0, -30.0, 0.0, 30.0, 60.0])
_probe_loss = paired_route_geometry_loss(_probe_routes, _probe_angles)
_probe_loss.backward()
assert torch.isfinite(_probe_loss)
assert _probe_logits.grad is not None and torch.isfinite(_probe_logits.grad).all()

# Router input-boundary gate.
from geometry_mmalls.model import ContextBottleneckRouter, SensoryBottleneckRouter
_context_router = ContextBottleneckRouter(2, 4, 64)
_sensory_router = SensoryBottleneckRouter(128, 4, 64)
assert _context_router(torch.randn(7, 2)).shape == (7, 4)
assert _sensory_router(torch.randn(7, 128)).shape == (7, 4)
assert sum(p.numel() for p in _context_router.parameters()) > 0
assert sum(p.numel() for p in _sensory_router.parameters()) > 0
print("Paired geometry backward gate: PASS")
print("Context/sensory bottleneck structural gate: PASS")

## 2. Controlled source split and deterministic loaders

In [ ]:
root = Path(config["data"]["root"])
base_train = MNIST(root=str(root), train=True, download=True)
base_test = MNIST(root=str(root), train=False, download=True)
rng = np.random.default_rng(SEED)
train_perm = rng.permutation(len(base_train))
test_perm = rng.permutation(len(base_test))
sensory_indices = train_perm[:SENSORY_SOURCE_LIMIT].tolist()
train_indices = train_perm[:TRAIN_SOURCE_LIMIT].tolist()
test_indices = test_perm[:TEST_SOURCE_LIMIT].tolist()
sensory_test_indices = test_perm[:SENSORY_TEST_LIMIT].tolist()

def sequence_checksum(values):
    return hashlib.sha256(",".join(map(str, values)).encode()).hexdigest()

def make_generator(seed):
    g = torch.Generator(); g.manual_seed(int(seed)); return g

def fixed_loader(angle, train, indices, shuffle, loader_seed=0, batch_size=128):
    ds = Subset(FixedAngleMNIST(root, angle=angle, train=train, download=True), indices)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0,
                      generator=make_generator(loader_seed) if shuffle else None)

def multi_loader(angles, train, indices, shuffle, loader_seed=0):
    ds = MultiAngleMNIST(root, angles=angles, train=train, indices=indices, download=True)
    return DataLoader(ds, batch_size=SOURCE_BATCH_SIZE, shuffle=shuffle, num_workers=0,
                      generator=make_generator(loader_seed) if shuffle else None)

def epoch_loader_seed(stage, epoch):
    return SEED * 1_000_000 + stage * 10_000 + epoch

split_manifest = {
    "seed": SEED,
    "sensory_source_count": len(sensory_indices),
    "train_source_count": len(train_indices),
    "test_source_count": len(test_indices),
    "sensory_index_sha256": sequence_checksum(sensory_indices),
    "train_index_sha256": sequence_checksum(train_indices),
    "test_index_sha256": sequence_checksum(test_indices),
}
split_manifest

## 3. Pretrain, evaluate, and freeze the sensory grove

In [ ]:
latent_dim = int(config["model"]["latent_dim"])
encoder = SmallConvEncoder(latent_dim).to(DEVICE)
sensory_head = torch.nn.Linear(latent_dim, 10).to(DEVICE)
optimizer = torch.optim.AdamW(
    list(encoder.parameters()) + list(sensory_head.parameters()),
    lr=float(config["training"]["learning_rate"]),
    weight_decay=float(config["training"]["weight_decay"]),
)
sensory_history=[]
for epoch in range(SENSORY_EPOCHS):
    loader = fixed_loader(0.0, True, sensory_indices, True, SEED*1000+epoch, 128)
    encoder.train(); sensory_head.train(); total=correct=0; loss_sum=0.0
    for x,y,_,_ in loader:
        x,y=x.to(DEVICE),y.to(DEVICE)
        logits=sensory_head(encoder(x)); loss=F.cross_entropy(logits,y)
        optimizer.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(list(encoder.parameters())+list(sensory_head.parameters()),5.0)
        optimizer.step()
        total += y.numel(); correct += (logits.argmax(1)==y).sum().item(); loss_sum += float(loss.detach())*y.numel()
    row={"epoch":epoch,"loss":loss_sum/total,"train_accuracy":correct/total}; sensory_history.append(row); print(row)

@torch.no_grad()
def evaluate_sensory(angle=0.0):
    encoder.eval(); sensory_head.eval(); total=correct=0; loss_sum=0.0
    for x,y,_,_ in fixed_loader(angle,False,sensory_test_indices,False,batch_size=256):
        x,y=x.to(DEVICE),y.to(DEVICE); logits=sensory_head(encoder(x))
        loss_sum += float(F.cross_entropy(logits,y,reduction="sum")); total += y.numel(); correct += (logits.argmax(1)==y).sum().item()
    return {"angle":angle,"accuracy":correct/total,"nll":loss_sum/total}

sensory_gate_result=evaluate_sensory(0.0)
print("Sensory gate:", sensory_gate_result, "threshold:", SENSORY_GATE)
if RUN_MODE=="qualification" and sensory_gate_result["accuracy"] < SENSORY_GATE:
    raise RuntimeError("Qualification blocked: sensory gate failed")
if sensory_gate_result["accuracy"] < SENSORY_GATE:
    print("DEVELOPMENT WARNING: sensory gate not reached; no C1-C6 promotion.")
sensory_state=copy.deepcopy(encoder.state_dict())

## 4. Shared initialization and declared route policies

In [ ]:
def build_model():
    local_encoder=SmallConvEncoder(latent_dim).to(DEVICE)
    local_encoder.load_state_dict(sensory_state)
    return GeometryMMALS(
        local_encoder,
        latent_dim=latent_dim,
        context_dim=int(config["model"]["context_dim"]),
        num_hosts=int(config["model"]["num_hosts"]),
        host_hidden_dim=int(config["model"]["host_hidden_dim"]),
        freeze_encoder=True,
        bottleneck_hidden_dim=64,
    ).to(DEVICE)

def state_dict_sha256(state_dict):
    digest=hashlib.sha256()
    for name in sorted(state_dict):
        t=state_dict[name].detach().cpu().contiguous()
        digest.update(name.encode()); digest.update(str(t.dtype).encode()); digest.update(str(tuple(t.shape)).encode()); digest.update(t.numpy().tobytes())
    return digest.hexdigest()

def forward_with_router_mode(model,x,router_mode="standard",context_override=None,z0_override=None,context_permutation=None,z0_permutation=None):
    z0=model.encode(x)
    context=model.context_net(z0)
    context_for_router = context_override if context_override is not None else (context[context_permutation] if context_permutation is not None else context)
    z0_for_router = z0_override if z0_override is not None else (z0[z0_permutation] if z0_permutation is not None else z0)
    if router_mode=="standard":
        route_logits=model.router(torch.cat([z0_for_router,context_for_router],dim=-1))
        route=torch.softmax(route_logits,dim=-1)
    elif router_mode=="context_bottleneck":
        route=torch.softmax(model.context_bottleneck_router(context_for_router),dim=-1)
    elif router_mode=="sensory_bottleneck":
        route=torch.softmax(model.sensory_bottleneck_router(z0_for_router),dim=-1)
    elif router_mode=="uniform_static":
        route=torch.full((z0.shape[0],len(model.hosts)),1.0/len(model.hosts),dtype=z0.dtype,device=z0.device)
    else:
        raise ValueError(f"Unknown router_mode: {router_mode}")
    return model.synthesize(z0,context,route)

def assert_finite_trace(trace,where):
    bad=[name for name,value in vars(trace).items() if torch.is_tensor(value) and not torch.isfinite(value).all()]
    if bad: raise FloatingPointError(f"Non-finite tensors at {where}: {bad}")

torch.manual_seed(SEED+107)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED+107)
base_model=build_model(); INITIAL_MMALS_STATE=copy.deepcopy(base_model.state_dict()); INITIAL_MMALS_SHA256=state_dict_sha256(INITIAL_MMALS_STATE)
router_parameter_counts={
    "standard":sum(p.numel() for p in base_model.router.parameters()),
    "context_bottleneck":sum(p.numel() for p in base_model.context_bottleneck_router.parameters()),
    "sensory_bottleneck":sum(p.numel() for p in base_model.sensory_bottleneck_router.parameters()),
    "uniform_static":0,
}
del base_model
print("Initial MMALS SHA256:",INITIAL_MMALS_SHA256)
print("Router parameter counts:",router_parameter_counts)

## 5. Equal-compute continual variants

In [ ]:
def host_diversity(host_outputs):
    h=F.normalize(host_outputs,dim=-1); sim=torch.einsum("bhd,bjd->bhj",h,h)
    mask=~torch.eye(sim.shape[-1],dtype=torch.bool,device=sim.device)
    return sim[:,mask].square().mean()

@torch.no_grad()
def evaluate_model(model,angles,stage,method,router_mode):
    rows=[]; model.eval()
    for angle in angles:
        total=correct=0; nll_sum=0.0
        for x,y,_,_ in fixed_loader(angle,False,test_indices,False,batch_size=256):
            x,y=x.to(DEVICE),y.to(DEVICE); tr=forward_with_router_mode(model,x,router_mode)
            assert_finite_trace(tr,f"eval {method} {angle}")
            total += y.numel(); correct += (tr.logits.argmax(1)==y).sum().item(); nll_sum += float(F.cross_entropy(tr.logits,y,reduction="sum"))
        rows.append({"method":method,"stage":stage,"angle":angle,"angle_type":"train" if angle in train_angles else ("interpolation" if angle in interp_angles else "extrapolation"),"accuracy":correct/total,"nll":nll_sum/total})
    return rows

def train_variant(method,spec,curriculum):
    model=build_model(); model.load_state_dict(INITIAL_MMALS_STATE,strict=True)
    assert state_dict_sha256(model.state_dict())==INITIAL_MMALS_SHA256
    trainable=[p for p in model.parameters() if p.requires_grad]
    optimizer=torch.optim.AdamW(trainable,lr=float(config["training"]["learning_rate"]),weight_decay=float(config["training"]["weight_decay"]))
    stage_rows=[]; eval_rows=[]
    for stage,current_angle in enumerate(curriculum):
        seen_angles=curriculum[:stage+1]
        totals={k:0.0 for k in ["loss","current_ce","anchor_ce","geo","div"]}; totals.update({"source_examples":0,"forward_images":0,"optimizer_steps":0})
        start=time.perf_counter(); model.train()
        for epoch in range(STAGE_EPOCHS):
            loader=multi_loader(seen_angles,True,train_indices,True,epoch_loader_seed(stage,epoch))
            for batch_id,(views,y,factors,_) in enumerate(loader):
                source_count,angle_count=views.shape[:2]
                flat=views.reshape(-1,*views.shape[2:]).to(DEVICE); y=y.to(DEVICE); factors=factors.to(DEVICE)
                tr=forward_with_router_mode(model,flat,spec["router_mode"]); assert_finite_trace(tr,f"{method} stage={stage} batch={batch_id}")
                logits=tr.logits.reshape(source_count,angle_count,-1); routes=tr.route.reshape(source_count,angle_count,-1)
                hosts=tr.host_outputs.reshape(source_count,angle_count,tr.host_outputs.shape[1],-1)
                current_ce=F.cross_entropy(logits[:,-1],y)
                if angle_count>1:
                    old_logits=logits[:,:-1].reshape(-1,logits.shape[-1]); old_targets=y[:,None].expand(-1,angle_count-1).reshape(-1)
                    anchor_ce=F.cross_entropy(old_logits,old_targets)
                else: anchor_ce=logits.sum()*0.0
                geo=paired_route_geometry_loss(routes,factors,bandwidth=float(config["losses"]["route_bandwidth_degrees"]),far_margin=float(config["losses"]["paired_route_far_margin"]),far_weight=float(config["losses"]["paired_route_far_weight"]),match_weight=float(config["losses"]["paired_route_match_weight"])) if spec["geometry_weight"]>0 else routes.sum()*0.0
                div=host_diversity(hosts[:,-1])
                loss=float(config["losses"]["classification"])*current_ce+spec["anchor_weight"]*anchor_ce+spec["geometry_weight"]*geo+float(config["losses"]["host_functional_diversity"])*div
                if not torch.isfinite(torch.stack([current_ce,anchor_ce,geo,div,loss])).all(): raise FloatingPointError(f"Non-finite loss {method}")
                optimizer.zero_grad(set_to_none=True); loss.backward(); grad=torch.nn.utils.clip_grad_norm_(trainable,5.0)
                if not torch.isfinite(torch.as_tensor(grad)): raise FloatingPointError("Non-finite gradient")
                optimizer.step()
                totals["source_examples"]+=source_count; totals["forward_images"]+=source_count*angle_count; totals["optimizer_steps"]+=1
                for key,val in [("loss",loss),("current_ce",current_ce),("anchor_ce",anchor_ce),("geo",geo),("div",div)]: totals[key]+=float(val.detach())*source_count
        count=max(totals["source_examples"],1)
        row={"method":method,"curriculum":json.dumps(curriculum),"stage":stage,"current_angle":current_angle,"seen_angles":json.dumps(seen_angles),"router_mode":spec["router_mode"],"geometry_weight":spec["geometry_weight"],"anchor_weight":spec["anchor_weight"],"source_examples":totals["source_examples"],"forward_images":totals["forward_images"],"optimizer_steps":totals["optimizer_steps"],"wall_seconds":time.perf_counter()-start,**{k:totals[k]/count for k in ["loss","current_ce","anchor_ce","geo","div"]}}
        stage_rows.append(row); print(row); eval_rows.extend(evaluate_model(model,all_eval_angles,stage,method,spec["router_mode"]))
    return model,pd.DataFrame(stage_rows),pd.DataFrame(eval_rows)

In [ ]:
models={}; stage_tables=[]; evaluation_tables=[]
for method in METHODS:
    print("\n=== TRAINING",method,"===")
    model,s,e=train_variant(method,METHOD_SPECS[method],PRIMARY_CURRICULUM)
    models[method]=model; stage_tables.append(s); evaluation_tables.append(e)
stage_df=pd.concat(stage_tables,ignore_index=True); evaluation_df=pd.concat(evaluation_tables,ignore_index=True)
stage_df

## 6. Compute, parameter-count, staged accuracy, and forgetting audits

In [ ]:
for stage in sorted(stage_df.stage.unique()):
    block=stage_df[stage_df.stage==stage]
    assert block.forward_images.nunique()==1
    assert block.optimizer_steps.nunique()==1
    assert block.source_examples.nunique()==1
compute_summary_df=stage_df.groupby("method").agg(total_forward_images=("forward_images","sum"),total_optimizer_steps=("optimizer_steps","sum"),total_source_examples=("source_examples","sum"),total_wall_seconds=("wall_seconds","sum")).reset_index()

def forgetting_table(evaluation,curriculum):
    rows=[]; final_stage=int(evaluation.stage.max())
    for method in evaluation.method.unique():
        sub=evaluation[(evaluation.method==method)&(evaluation.angle_type=="train")]
        for angle in train_angles:
            seen=curriculum.index(angle); r=sub[(sub.angle==angle)&(sub.stage>=seen)].sort_values("stage")
            best=float(r.accuracy.max()); final=float(r[r.stage==final_stage].accuracy.iloc[0])
            rows.append({"method":method,"angle":angle,"best_accuracy":best,"final_accuracy":final,"forgetting":best-final})
    return pd.DataFrame(rows)
forgetting_df=forgetting_table(evaluation_df,PRIMARY_CURRICULUM)
print("Equal paired compute audit: PASS")
compute_summary_df, forgetting_df.groupby("method").agg(mean_forgetting=("forgetting","mean"),final_trained_accuracy=("final_accuracy","mean"))

## 7. Same-source traces and partitioned geometry

In [ ]:
@torch.no_grad()
def collect_paired_trace(model,method,angles,source_indices,router_mode):
    model.eval(); rows=[]
    for views,labels,factors,source_ids in multi_loader(angles,False,source_indices,False):
        b,a=views.shape[:2]; flat=views.reshape(-1,*views.shape[2:]).to(DEVICE)
        tr=forward_with_router_mode(model,flat,router_mode); assert_finite_trace(tr,f"trace {method}")
        arrays={"z0":tr.z0.reshape(b,a,-1).cpu().numpy(),"context":tr.context.reshape(b,a,-1).cpu().numpy(),"route":tr.route.reshape(b,a,-1).cpu().numpy(),"z_mm":tr.z_mm.reshape(b,a,-1).cpu().numpy(),"logits":tr.logits.reshape(b,a,-1).cpu().numpy(),"hosts":tr.host_outputs.reshape(b,a,tr.host_outputs.shape[1],-1).cpu().numpy()}
        for i in range(b):
            for j in range(a):
                rows.append({"method":method,"source_index":int(source_ids[i]),"label":int(labels[i]),"angle":float(factors[i,j]),"prediction":int(np.argmax(arrays["logits"][i,j])),"correct":int(np.argmax(arrays["logits"][i,j])==int(labels[i])),"z0":arrays["z0"][i,j],"context":arrays["context"][i,j],"route":arrays["route"][i,j],"z_mm":arrays["z_mm"][i,j],"hosts":arrays["hosts"][i,j]})
    return pd.DataFrame(rows)
trace_df=pd.concat([collect_paired_trace(models[m],m,all_eval_angles,test_indices,METHOD_SPECS[m]["router_mode"]) for m in METHODS],ignore_index=True)
print("trace rows:",len(trace_df))

def stack_column(frame,name): return np.stack(frame[name].to_numpy())
def geometry_partition(frame,partition_name,angles):
    rows=[]; part=frame[frame.angle.isin(angles)]
    for method in METHODS:
        sub=part[part.method==method]; factors=sub.angle.to_numpy(float); groups=sub.source_index.to_numpy()
        for space,metric,reps in [("sensory","euclidean",stack_column(sub,"z0")),("context","euclidean",stack_column(sub,"context")),("route","fisher_rao",stack_column(sub,"route")),("synthesis","euclidean",stack_column(sub,"z_mm"))]:
            grouped=grouped_geometry_scores(factors,reps,groups,metric=metric); rm,rl,rh=bootstrap_mean_ci(grouped["rho"],BOOTSTRAP_SAMPLES,seed=SEED); sm,sl,sh=bootstrap_mean_ci(grouped["stress"],BOOTSTRAP_SAMPLES,seed=SEED+1); centroid=centroid_geometry_scores(factors,reps,metric=metric)
            rows.append({"partition":partition_name,"method":method,"space":space,"source_rho_mean":rm,"source_rho_ci_low":rl,"source_rho_ci_high":rh,"source_stress_mean":sm,"source_stress_ci_low":sl,"source_stress_ci_high":sh,"centroid_rho":centroid["rho"],"centroid_stress":centroid["stress"],"source_blocks":len(grouped["rho"]),"status":"development_not_qualified"})
    return pd.DataFrame(rows)
geometry_df=pd.concat([geometry_partition(trace_df,"C1_trained",train_angles),geometry_partition(trace_df,"C2_interpolation",interp_angles),geometry_partition(trace_df,"extrapolation",extra_angles)],ignore_index=True)
geometry_df

## 8. Paired source-level contrasts and held-out prediction

In [ ]:
def paired_bootstrap_delta(values,samples,seed):
    x=np.asarray(values,dtype=float); x=x[np.isfinite(x)]
    if len(x)==0:return (float("nan"),)*3
    rng=np.random.default_rng(seed); boot=np.array([rng.choice(x,size=len(x),replace=True).mean() for _ in range(samples)])
    return float(x.mean()),float(np.quantile(boot,0.025)),float(np.quantile(boot,0.975))

def source_geometry_frame(method,angles,space,metric):
    sub=trace_df[(trace_df.method==method)&(trace_df.angle.isin(angles))]
    g=grouped_geometry_scores(sub.angle.to_numpy(float),stack_column(sub,space),sub.source_index.to_numpy(),metric=metric)
    return pd.DataFrame({"source_index":g["group_ids"].astype(int),"rho":g["rho"],"stress":g["stress"]})

PAIRED_CONTRASTS=[
    ("bottleneck_geo_effect","context_bottleneck_geo","context_bottleneck_no_geo"),
    ("bottleneck_vs_standard","context_bottleneck_geo","standard_anchor_no_geo"),
    ("sensory_geo_effect","sensory_bottleneck_geo","sensory_bottleneck_no_geo"),
    ("context_vs_sensory_geo","context_bottleneck_geo","sensory_bottleneck_geo"),
    ("bottleneck_vs_uniform","context_bottleneck_geo","uniform_static_anchor"),
]
rows=[]
for contrast,treatment,control in PAIRED_CONTRASTS:
    for partition,angles in [("C1_trained",train_angles),("C2_interpolation",interp_angles)]:
        for space,metric in [("context","euclidean"),("route","fisher_rao"),("z_mm","euclidean")]:
            t=source_geometry_frame(treatment,angles,space,metric); c=source_geometry_frame(control,angles,space,metric); m=t.merge(c,on="source_index",suffixes=("_t","_c"))
            dm,dl,dh=paired_bootstrap_delta(m.rho_t-m.rho_c,BOOTSTRAP_SAMPLES,SEED+20)
            sm,sl,sh=paired_bootstrap_delta(m.stress_t-m.stress_c,BOOTSTRAP_SAMPLES,SEED+21)
            rows.append({"contrast":contrast,"treatment":treatment,"control":control,"partition":partition,"space":"synthesis" if space=="z_mm" else space,"paired_source_count":len(m),"delta_rho_mean":dm,"delta_rho_ci_low":dl,"delta_rho_ci_high":dh,"delta_stress_mean":sm,"delta_stress_ci_low":sl,"delta_stress_ci_high":sh,"status":"development_not_qualified"})
paired_source_delta_df=pd.DataFrame(rows)

final_stage=int(evaluation_df.stage.max())
interpolation_df=evaluation_df[(evaluation_df.stage==final_stage)&(evaluation_df.angle_type=="interpolation")].copy()
pred_rows=[]
for contrast,treatment,control in PAIRED_CONTRASTS:
    t=trace_df[(trace_df.method==treatment)&(trace_df.angle.isin(interp_angles))][["source_index","angle","correct"]]
    c=trace_df[(trace_df.method==control)&(trace_df.angle.isin(interp_angles))][["source_index","angle","correct"]]
    m=t.merge(c,on=["source_index","angle"],suffixes=("_t","_c"))
    for angle,block in m.groupby("angle"):
        dm,dl,dh=paired_bootstrap_delta(block.correct_t-block.correct_c,BOOTSTRAP_SAMPLES,SEED+40+int(angle+90))
        pred_rows.append({"contrast":contrast,"treatment":treatment,"control":control,"angle":angle,"delta_accuracy_mean":dm,"delta_accuracy_ci_low":dl,"delta_accuracy_ci_high":dh,"paired_source_count":len(block),"status":"development_not_qualified"})
paired_prediction_delta_df=pd.DataFrame(pred_rows)
paired_source_delta_df, paired_prediction_delta_df

## 9. Mediation interventions: is context now primary?

In [ ]:
@torch.no_grad()
def mediation_intervention_table(model,method,router_mode,angles,source_indices):
    raw_rows=[]; model.eval()
    for angle in angles:
        for x,y,_,source_ids in fixed_loader(angle,False,source_indices,False,batch_size=256):
            x,y=x.to(DEVICE),y.to(DEVICE); base=forward_with_router_mode(model,x,router_mode)
            perm=torch.arange(len(y)-1,-1,-1,device=DEVICE)
            interventions={
                "standard":base,
                "router_context_zeroed":forward_with_router_mode(model,x,router_mode,context_override=torch.zeros_like(base.context)),
                "router_context_shuffled":forward_with_router_mode(model,x,router_mode,context_permutation=perm),
                "router_z0_zeroed":forward_with_router_mode(model,x,router_mode,z0_override=torch.zeros_like(base.z0)),
                "router_z0_shuffled":forward_with_router_mode(model,x,router_mode,z0_permutation=perm),
            }
            base_correct=(base.logits.argmax(1)==y).float(); base_logp=F.log_softmax(base.logits,-1).gather(1,y[:,None]).squeeze(1)
            for name,tr in interventions.items():
                route_shift=torch.linalg.vector_norm(tr.route-base.route,dim=1); correct=(tr.logits.argmax(1)==y).float(); logp=F.log_softmax(tr.logits,-1).gather(1,y[:,None]).squeeze(1)
                for i in range(len(y)):
                    raw_rows.append({"method":method,"router_mode":router_mode,"angle":angle,"source_index":int(source_ids[i]),"intervention":name,"accuracy":float(correct[i]),"accuracy_delta_vs_standard":float(correct[i]-base_correct[i]),"route_shift":float(route_shift[i]),"class_log_prob_drop":float(base_logp[i]-logp[i])})
    raw=pd.DataFrame(raw_rows)
    summary=raw.groupby(["method","router_mode","intervention"]).agg(accuracy=("accuracy","mean"),accuracy_delta_vs_standard=("accuracy_delta_vs_standard","mean"),mean_route_shift=("route_shift","mean"),mean_class_log_prob_drop=("class_log_prob_drop","mean"),count=("source_index","count")).reset_index()
    def s(name,col): return float(summary[summary.intervention==name][col].iloc[0])
    context_shift=np.mean([s("router_context_zeroed","mean_route_shift"),s("router_context_shuffled","mean_route_shift")])
    z0_shift=np.mean([s("router_z0_zeroed","mean_route_shift"),s("router_z0_shuffled","mean_route_shift")])
    context_drop=-s("router_context_zeroed","accuracy_delta_vs_standard")
    z0_drop=-s("router_z0_zeroed","accuracy_delta_vs_standard")
    dependency=pd.DataFrame([{"method":method,"router_mode":router_mode,"context_route_shift_mean":context_shift,"z0_route_shift_mean":z0_shift,"context_to_z0_shift_ratio":context_shift/max(z0_shift,1e-12),"context_accuracy_drop":context_drop,"z0_accuracy_drop":z0_drop,"context_is_primary":bool(context_shift>z0_shift and context_drop>z0_drop),"status":"development_not_qualified"}])
    return raw,summary,dependency

med_raw=[]; med_summary=[]; med_dep=[]
for method in METHODS:
    r,s,d=mediation_intervention_table(models[method],method,METHOD_SPECS[method]["router_mode"],train_angles+interp_angles,test_indices)
    med_raw.append(r); med_summary.append(s); med_dep.append(d)
mediation_raw_df=pd.concat(med_raw,ignore_index=True); mediation_summary_df=pd.concat(med_summary,ignore_index=True); mediation_dependency_df=pd.concat(med_dep,ignore_index=True)
mediation_dependency_df

## 10. Signed causal direction probe

In [ ]:
def orthogonal_directions(tangent,count,seed):
    rng=np.random.default_rng(seed); directions=[]
    while len(directions)<count:
        v=rng.normal(size=tangent.shape); v=v-np.dot(v,tangent)*tangent; n=np.linalg.norm(v)
        if n>1e-12: directions.append(v/n)
    return directions

@torch.no_grad()
def signed_causal_probe(model,method,router_mode,trace,probe_angle=15.0,scales=(-2,-1,0,1,2)):
    trained=trace[trace.angle.isin(train_angles)]; contexts=stack_column(trained,"context").astype(np.float64); angles=trained.angle.to_numpy(float)
    x=contexts-contexts.mean(0,keepdims=True); y=angles-angles.mean(); ridge=1e-6*max(np.trace(x.T@x)/max(x.shape[1],1),1.0)
    tangent=np.linalg.solve(x.T@x+ridge*np.eye(x.shape[1]),x.T@y); tangent=tangent/max(np.linalg.norm(tangent),1e-12)
    centroids={float(a):np.stack(g.route).mean(axis=0) for a,g in trained.groupby("angle")}; rm=np.stack([np.sqrt(np.clip(centroids[a],1e-12,None)) for a in train_angles]); ca=np.asarray(train_angles)-np.mean(train_angles); rc=rm-rm.mean(0,keepdims=True); rd=(ca[:,None]*rc).sum(0)/max(np.sum(ca**2),1e-12); rd=rd/max(np.linalg.norm(rd),1e-12)
    xb,yb,_,_=next(iter(fixed_loader(probe_angle,False,test_indices,False,batch_size=128))); xb,yb=xb.to(DEVICE),yb.to(DEVICE); base=forward_with_router_mode(model,xb,router_mode); base_logp=F.log_softmax(base.logits,-1).gather(1,yb[:,None]).squeeze(1)
    tt=torch.tensor(tangent,dtype=base.context.dtype,device=DEVICE); rt=torch.tensor(rd,dtype=base.route.dtype,device=DEVICE); orths=orthogonal_directions(tangent,8,SEED+9); rows=[]
    for scale in scales:
        tr=forward_with_router_mode(model,xb,router_mode,context_override=base.context+float(scale)*tt); delta=torch.sqrt(tr.route.clamp_min(1e-12))-torch.sqrt(base.route.clamp_min(1e-12)); signed=float((delta*rt).sum(1).mean().cpu()); new_logp=F.log_softmax(tr.logits,-1).gather(1,yb[:,None]).squeeze(1); identity=float((base_logp-new_logp).mean().cpu())
        oe=[]
        for d in orths:
            dt=torch.tensor(d,dtype=base.context.dtype,device=DEVICE); o=forward_with_router_mode(model,xb,router_mode,context_override=base.context+float(scale)*dt); dd=torch.sqrt(o.route.clamp_min(1e-12))-torch.sqrt(base.route.clamp_min(1e-12)); oe.append(float(torch.abs((dd*rt).sum(1)).mean().cpu()))
        om=float(np.mean(oe)); csr=abs(signed)/max(om,1e-12) if scale else 0.0; orientation=True if scale==0 else bool(np.sign(signed)==np.sign(scale))
        rows.append({"method":method,"probe_angle":probe_angle,"scale":scale,"signed_route_effect":signed,"orthogonal_abs_effect_mean":om,"causal_specificity_ratio":csr,"orientation_match":orientation,"class_log_prob_drop":identity,"status":"development_not_qualified"})
    frame=pd.DataFrame(rows); abs1=frame[frame.scale.abs()==1].signed_route_effect.abs().mean(); abs2=frame[frame.scale.abs()==2].signed_route_effect.abs().mean()
    summary=pd.DataFrame([{"method":method,"mean_csr_nonzero":frame[frame.scale!=0].causal_specificity_ratio.mean(),"orientation_pass":bool(frame[frame.scale!=0].orientation_match.all()),"monotonic_magnitude_pass":bool(abs2>abs1),"max_abs_class_log_prob_drop":frame.class_log_prob_drop.abs().max(),"candidate_causal_gate":bool(frame[frame.scale!=0].orientation_match.all() and abs2>abs1 and frame[frame.scale!=0].causal_specificity_ratio.mean()>1.5 and frame.class_log_prob_drop.abs().max()<0.05),"status":"development_not_qualified"}])
    return frame,summary

causal_raw=[]; causal_summary=[]
for method in ["standard_anchor_no_geo","context_bottleneck_no_geo","context_bottleneck_geo","sensory_bottleneck_no_geo","sensory_bottleneck_geo"]:
    r,s=signed_causal_probe(models[method],method,METHOD_SPECS[method]["router_mode"],trace_df[trace_df.method==method]); causal_raw.append(r); causal_summary.append(s)
causal_df=pd.concat(causal_raw,ignore_index=True); causal_summary_df=pd.concat(causal_summary,ignore_index=True)
causal_summary_df

## 11. Host route-mass and functional-ablation bundle

In [ ]:
@torch.no_grad()
def host_bundle(model,method,router_mode,angles,source_indices):
    rows=[]; model.eval()
    for angle in angles:
        for x,y,_,sid in fixed_loader(angle,False,source_indices,False,batch_size=256):
            x,y=x.to(DEVICE),y.to(DEVICE); tr=forward_with_router_mode(model,x,router_mode); base=F.log_softmax(tr.logits,-1).gather(1,y[:,None]).squeeze(1)
            for h in range(tr.route.shape[1]):
                route=tr.route.clone(); route[:,h]=0; route=route/route.sum(1,keepdim=True).clamp_min(1e-8); z=torch.einsum("bh,bhd->bd",route,tr.host_outputs); logits=model.classifier(model.synthesis_norm(z)); abl=F.log_softmax(logits,-1).gather(1,y[:,None]).squeeze(1); impact=(base-abl).cpu().numpy(); mass=tr.route[:,h].cpu().numpy()
                for i,v in enumerate(impact): rows.append({"method":method,"angle":angle,"source_index":int(sid[i]),"host":h,"route_mass":float(mass[i]),"ablation_impact":float(v)})
    raw=pd.DataFrame(rows); summary=raw.groupby(["method","angle","host"]).agg(route_mass_mean=("route_mass","mean"),ablation_mean=("ablation_impact","mean"),ablation_median=("ablation_impact","median"),positive_fraction=("ablation_impact",lambda x:float(np.mean(x>0))),count=("ablation_impact","count")).reset_index(); return raw,summary
host_raw=[];host_summary=[]
for method in METHODS:
    r,s=host_bundle(models[method],method,METHOD_SPECS[method]["router_mode"],train_angles+interp_angles,test_indices); host_raw.append(r);host_summary.append(s)
host_raw_df=pd.concat(host_raw,ignore_index=True);host_summary_df=pd.concat(host_summary,ignore_index=True)
host_summary_df.head()

## 12. Optional same-final-task curriculum reruns for the primary bottleneck method

In [ ]:
curriculum_stage_tables=[]; curriculum_eval_tables=[]
primary_method="context_bottleneck_geo"
curriculum_stage_tables.append(stage_df[stage_df.method==primary_method].assign(curriculum_name=PRIMARY_CURRICULUM_NAME))
curriculum_eval_tables.append(evaluation_df[evaluation_df.method==primary_method].assign(curriculum_name=PRIMARY_CURRICULUM_NAME))
if RUN_COUNTERBALANCED_CURRICULA:
    for name,curriculum in COUNTERBALANCED_CURRICULA.items():
        if name==PRIMARY_CURRICULUM_NAME: continue
        method=f"context_bottleneck_geo_{name}"; print("\n=== CURRICULUM",name,"===")
        model,s,e=train_variant(method,METHOD_SPECS[primary_method],curriculum); curriculum_stage_tables.append(s.assign(curriculum_name=name)); curriculum_eval_tables.append(e.assign(curriculum_name=name))
curriculum_stage_df=pd.concat(curriculum_stage_tables,ignore_index=True); curriculum_evaluation_df=pd.concat(curriculum_eval_tables,ignore_index=True)
curriculum_summary=[]
for name,curriculum in COUNTERBALANCED_CURRICULA.items():
    sub=curriculum_evaluation_df[curriculum_evaluation_df.curriculum_name==name]
    if len(sub)==0: continue
    f=forgetting_table(sub,curriculum); final=sub[sub.stage==sub.stage.max()]
    curriculum_summary.append({"curriculum":name,"order":json.dumps(curriculum),"mean_forgetting":f.forgetting.mean(),"final_trained_accuracy":final[final.angle_type=="train"].accuracy.mean(),"mean_interpolation_accuracy":final[final.angle_type=="interpolation"].accuracy.mean(),"status":"development_not_qualified"})
curriculum_summary_df=pd.DataFrame(curriculum_summary)
curriculum_summary_df

## 13. Development summaries and explicit gates

In [ ]:
def geo_value(method,space,field="source_rho_mean"):
    r=geometry_df[(geometry_df.partition=="C1_trained")&(geometry_df.method==method)&(geometry_df.space==space)]
    return float(r[field].iloc[0])
def method_summary(method):
    final=evaluation_df[(evaluation_df.method==method)&(evaluation_df.stage==evaluation_df.stage.max())]; f=forgetting_df[forgetting_df.method==method]; c=causal_summary_df[causal_summary_df.method==method]; dep=mediation_dependency_df[mediation_dependency_df.method==method].iloc[0]
    return {"method":method,"router_mode":METHOD_SPECS[method]["router_mode"],"trained_context_source_rho":geo_value(method,"context"),"trained_route_source_rho":geo_value(method,"route"),"trained_route_centroid_rho":geo_value(method,"route","centroid_rho"),"mean_interpolation_accuracy":final[final.angle_type=="interpolation"].accuracy.mean(),"final_trained_accuracy":final[final.angle_type=="train"].accuracy.mean(),"mean_forgetting":f.forgetting.mean(),"context_to_z0_shift_ratio":dep.context_to_z0_shift_ratio,"context_accuracy_drop":dep.context_accuracy_drop,"z0_accuracy_drop":dep.z0_accuracy_drop,"context_is_primary":dep.context_is_primary,"mean_csr":float(c.mean_csr_nonzero.iloc[0]) if len(c) else float("nan"),"qualification_status":"not_qualified"}
method_summary_df=pd.DataFrame([method_summary(m) for m in METHODS])

def contrast_row(name,t,c):
    tm=method_summary_df[method_summary_df.method==t].iloc[0]; cm=method_summary_df[method_summary_df.method==c].iloc[0]
    return {"contrast":name,"treatment":t,"control":c,"delta_context_rho":tm.trained_context_source_rho-cm.trained_context_source_rho,"delta_route_rho":tm.trained_route_source_rho-cm.trained_route_source_rho,"delta_interpolation_accuracy":tm.mean_interpolation_accuracy-cm.mean_interpolation_accuracy,"delta_final_trained_accuracy":tm.final_trained_accuracy-cm.final_trained_accuracy,"delta_forgetting":tm.mean_forgetting-cm.mean_forgetting,"status":"development_not_qualified"}
contrast_summary_df=pd.DataFrame([contrast_row(*x) for x in PAIRED_CONTRASTS])

gate_summary_df=pd.DataFrame([
    {"gate":"C0_protocol_integrity","status":"candidate_pass_after_successful_execution","requirement":"all finite-value, compute, package, split and export assertions pass"},
    {"gate":"context_mediation_architectural","status":"not_qualified","requirement":"context_bottleneck_geo context_is_primary and ratio > 1 across seeds"},
    {"gate":"C1_context_geometry","status":"not_qualified","requirement":"bottleneck_geo_effect context rho CI lower > 0 across seeds"},
    {"gate":"C2_interpolation","status":"not_qualified","requirement":"paired prediction and geometry gains across seeds"},
    {"gate":"C3_host_specialization","status":"not_qualified","requirement":"locality, ablation and cross-seed role matching"},
    {"gate":"C4_causal_geometry","status":"not_qualified","requirement":"CSR > 1.5, orientation, monotonicity and identity preservation"},
    {"gate":"C5_memory_transport","status":"not_implemented","requirement":"reconstructive and synthetic memory transport"},
    {"gate":"C6_operational_utility","status":"not_qualified","requirement":"accuracy-retention-cost advantage over sensory and static controls"},
])
method_summary_df,contrast_summary_df,gate_summary_df

## 14. Export and immutable manifest

In [ ]:
try: git_sha=subprocess.check_output(["git","rev-parse","HEAD"],text=True).strip()
except Exception: git_sha="unknown"
source_hashes={}
for relative in ["pyproject.toml","src/geometry_mmalls/__init__.py","src/geometry_mmalls/model.py","src/geometry_mmalls/geometry.py","src/geometry_mmalls/metrics.py","configs/rotated_mnist_g1_v107.yaml"]:
    p=REPO_DIR/relative; source_hashes[relative]=hashlib.sha256(p.read_bytes()).hexdigest()
output_root=Path("results/v1_0_7")/f"seed_{SEED}"; output_root.mkdir(parents=True,exist_ok=True)
exports={
    "stage_logs.csv":stage_df,"compute_summary.csv":compute_summary_df,"staged_evaluation.csv":evaluation_df,"forgetting.csv":forgetting_df,"partitioned_geometry_metrics.csv":geometry_df,"paired_source_deltas.csv":paired_source_delta_df,"interpolation_metrics.csv":interpolation_df,"paired_prediction_deltas.csv":paired_prediction_delta_df,"mediation_interventions_raw.csv":mediation_raw_df,"mediation_interventions_summary.csv":mediation_summary_df,"mediation_dependency_summary.csv":mediation_dependency_df,"causal_probe_raw.csv":causal_df,"causal_probe_summary.csv":causal_summary_df,"host_bundle_raw.csv":host_raw_df,"host_bundle_summary.csv":host_summary_df,"curriculum_stage_logs.csv":curriculum_stage_df,"curriculum_evaluation.csv":curriculum_evaluation_df,"curriculum_summary.csv":curriculum_summary_df,"development_method_summary.csv":method_summary_df,"development_contrasts.csv":contrast_summary_df,"development_gate_summary.csv":gate_summary_df,"sensory_history.csv":pd.DataFrame(sensory_history),"sensory_gate.csv":pd.DataFrame([sensory_gate_result]),
}
for filename,frame in exports.items(): frame.to_csv(output_root/filename,index=False)
TRACKED_CHANGES_107=[f"CHG-107-{i:02d}" for i in range(1,13)]
run_manifest={
    "status":"development_not_qualified","notebook_version":NOTEBOOK_VERSION,"package_version":geometry_mmalls.__version__,"release_integrity_passed":geometry_mmalls.__version__==NOTEBOOK_VERSION,"git_sha":git_sha,"source_hashes":source_hashes,"initial_mmalls_sha256":INITIAL_MMALS_SHA256,"router_parameter_counts":router_parameter_counts,"run_mode":RUN_MODE,"seed":SEED,"device":str(DEVICE),"method_specs":METHOD_SPECS,"primary_curriculum_name":PRIMARY_CURRICULUM_NAME,"counterbalanced_curricula":COUNTERBALANCED_CURRICULA,"split_manifest":split_manifest,"sensory_gate":{"threshold":SENSORY_GATE,"result":sensory_gate_result,"passed":sensory_gate_result["accuracy"]>=SENSORY_GATE},"tracked_changes":TRACKED_CHANGES_107,"reviewer_status_report":"docs/reports/Geometry_MMALS_G1_Status_and_Perspective_Reviewer_Report_v1_0.pdf","reviewer_latex_source":"paper/reviewer_status_v1_0/","warning":"Context-bottleneck protocol only. C1-C6 remain unqualified.","timestamp":time.time(),
}
(output_root/"run_manifest.json").write_text(json.dumps(run_manifest,indent=2),encoding="utf-8")
try: (output_root/"environment.txt").write_text(subprocess.check_output([sys.executable,"-m","pip","freeze"],text=True),encoding="utf-8")
except Exception as exc: print("Environment export skipped:",exc)
print("Exported to:",output_root.resolve())

## 15. Reviewer interpretation matrix

### Scenario A - architectural shortcut confirmed

Required together:

- `context_bottleneck_geo.context_is_primary == True`;
- context-to-z0 route-shift ratio above 1;
- positive context accuracy drop when context is zeroed;
- `bottleneck_geo_effect` context-order confidence interval above zero.

Interpretation: the v1.0.6 context failure was primarily architectural. The next step is a controlled soft bottleneck or learned mediation gate.

### Scenario B - representation remains insufficient

- context bottleneck is context-dependent by construction but geometry does not improve;
- prediction remains close to static routing;
- causal specificity remains below threshold.

Interpretation: the context representation or its supervision must be redesigned before G2 energy routing or G3 phase-aware routing.

### Scenario C - geometry/accuracy trade-off

- context order improves under the hard bottleneck;
- accuracy drops relative to sensory-only or standard routing.

Interpretation: context contains partial geometric structure but is not yet sufficient for host selection. Test a regularized soft gate only after this result is replicated.

## 16. Qualification checklist

Before any C1-C6 promotion:

1. Freeze numeric gates before qualification runs.
2. Require sensory acceptance.
3. Run at least five pilot seeds and ten final-evidence seeds.
4. Keep source split, initialization, curriculum, forwards and optimizer steps matched.
5. Preserve the capacity-matched sensory bottleneck control.
6. Report failed seeds and source-paired confidence intervals.
7. Confirm held-out angles never enter training or checkpoint selection.
8. Require context mediation and causal specificity together; neither is sufficient alone.
9. Add cross-seed host-role matching before C3.
10. Replace the direct old-view anchor with reconstructive/synthetic memory before C5.
11. Replicate on a second controlled transformation.
12. Archive notebook, CSVs, manifest, environment, exact commit, reviewer report, and LaTeX source.

## 17. Optional notebook PDF export

In [ ]:
# The clean protocol PDF under docs/reports is preferred for GitHub.
# This optional cell avoids recursive Drive scans and exports only an explicit file.
from pathlib import Path
notebook_candidates=[
    Path("/content/Geometry_MMALS_G1_ContextBottleneck_v1_0_7.ipynb"),
    Path("/content/drive/MyDrive/Colab Notebooks/Geometry_MMALS_G1_ContextBottleneck_v1_0_7.ipynb"),
]
notebook_path=next((p for p in notebook_candidates if p.exists()),None)
if notebook_path is None:
    print("Notebook file not found at declared locations; export skipped.")
else:
    output_dir=Path("/content/pdf_export"); output_dir.mkdir(parents=True,exist_ok=True)
    command=[sys.executable,"-m","jupyter","nbconvert","--to","pdf",str(notebook_path),"--output-dir",str(output_dir),"--PDFExporter.latex_command=xelatex","--PDFExporter.latex_command={filename}","--PDFExporter.latex_command=-interaction=nonstopmode","--PDFExporter.latex_count=3"]
    print(" ".join(command))
    # subprocess.run(command, check=True)